# SmolVLM2 vision-block transfer pilot

This notebook runs the matched one-seed screening pilot for the 814M LFM-aligned document VLM. The control uses strict LFM language initialization; the treatment adds exact SmolVLM2 transformer blocks. It requires native CUDA BF16, such as L4, A10, A100, or newer. T4 is intentionally rejected. A passing run is screening evidence only and does not authorize promotion.

In [ ]:
import os, shutil, subprocess, sys
from pathlib import Path

ROOT = next((p for p in (Path.cwd(), Path.cwd().parent) if (p / 'pyproject.toml').is_file()), None)
if ROOT is None:
    subprocess.run(['git', 'clone', 'https://github.com/SangbumChoi/OCR.git'], check=True)
    ROOT = Path('OCR').resolve()
subprocess.run(['git', '-C', str(ROOT), 'fetch', 'origin', 'claude/new-session-w79q0i'], check=True)
subprocess.run(['git', '-C', str(ROOT), 'checkout', 'claude/new-session-w79q0i'], check=True)
subprocess.run(['git', '-C', str(ROOT), 'merge', '--ff-only', 'origin/claude/new-session-w79q0i'], check=True)
os.chdir(ROOT)
if shutil.which('apt-get'):
    subprocess.run(['apt-get', 'update', '-qq'], check=True)
    subprocess.run(['apt-get', 'install', '-y', '-qq', 'libpango-1.0-0', 'libpangoft2-1.0-0', 'libharfbuzz0b', 'libfontconfig1', 'fonts-liberation', 'fonts-noto-core', 'fonts-noto-cjk'], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-e', '.[student,student-gpu,newvlms,synth,finetune]'], check=True)
print('repo:', ROOT)


In [ ]:
import os
import wandb

if not os.environ.get('WANDB_API_KEY'):
    try:
        from google.colab import userdata
        key = userdata.get('WANDB_API_KEY')
        if key:
            os.environ['WANDB_API_KEY'] = key
    except Exception:
        pass
wandb.login()
print('W&B target: https://wandb.ai/sbdc/docvlm-ablation')


In [ ]:
# Recompute all 14 readiness checks without allocating the student checkpoint.
subprocess.run([sys.executable, 'scripts/run_transfer_pilot_colab.py', '--pilot', 'smol-vision', '--dry-run', '--poll-seconds', '0.1'], check=True)


In [ ]:
# Run both matched cells. Resume skips every signature-valid completed stage.
subprocess.run([sys.executable, 'scripts/run_transfer_pilot_colab.py', '--pilot', 'smol-vision'], check=True)


In [ ]:
import json
from pathlib import Path

summary_path = Path('outputs/sweeps/docvlm-smol-vision-transfer-pilot/sweep_run_summary.json')
summary = json.loads(summary_path.read_text())
if summary.get('status') != 'completed':
    raise RuntimeError('pilot summary is not completed')
subprocess.run([sys.executable, 'scripts/publish_smol_pilot_handoff.py'], check=True)
subprocess.run([sys.executable, 'scripts/snapshot_wandb_run_inventory.py'], check=True)
subprocess.run([sys.executable, 'scripts/audit_smol_vision_transfer_pilot_execution.py'], check=True)
print(json.dumps({
    'status': summary.get('status'),
    'variants': [{'run': row.get('run'), 'status': row.get('status')} for row in summary.get('variants', [])],
    'comparison': summary.get('comparison'),
}, indent=2))
print('W&B: https://wandb.ai/sbdc/docvlm-ablation')
